# Generation metrics on a Cohere model: Tiny Aya

This notebook shows AuditKIT's **generation metrics** on one real model, **Tiny Aya Global** (`cohere2`, 3.3B) on the `hf:` backend, with every number coming from live generations:

| § | Task | Metrics |
|---|---|---|
| 1 | Multilingual short answers | `exact_match`, `quasi_exact_match`, `f1_score`, `levenshtein` |
| 2 | Translation | `bleu`, `chrf`, `rouge_l`, `token_overlap`, `bert_score` (optional) |
| 3 | Format following | `is_json`, `regex`, `word_count`, `starts_with` |
| 4 | Math word problems (GSM8K slice) | answer extraction plus a custom last-number metric |
| 5 | Performance | latency and throughput, recorded on every run |

**Runtime:** a Colab **T4 is enough**: Tiny Aya is ~6.7 GB in fp16. **Access:** gated, so accept the licence and add the `HF_TOKEN` secret, plus `GITHUB_TOKEN` to install AuditKIT.

In [ ]:
# ---- settings ----
import os
MODEL = os.environ.get("NB_MODEL", "CohereLabs/tiny-aya-global")
MAX_TOKENS = 96
GSM8K_N = int(os.environ.get("NB_GSM8K_N", "20"))
USE_BERT_SCORE = False       # downloads a RoBERTa model; set True for a semantic metric

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at REPO_BRANCH and installs it with the extras this notebook needs.
# Local: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, sys, subprocess
REPO_BRANCH = "feat/rag-agent-evals"   # #1's branch: #9, #10 and every #15 / #27 fix are merged here
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ("GITHUB_TOKEN", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", REPO_BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[transformers,vision]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
else:
    print("HF_TOKEN not set: gated Cohere models (Tiny Aya, Aya Expanse, Aya Vision) will fail to download.")

In [ ]:
import auditkit as ak, torch, transformers
from auditkit.runspec import RunConfig
from auditkit.model.hf_gen import HFGenModel
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if (BF16 or DEVICE == "mps") else torch.float16
print(f"auditkit {ak.__version__} | transformers {transformers.__version__} | torch {torch.__version__}")
print(f"device {DEVICE} | GPU {GPU_GB:.0f} GB | dtype {DTYPE}")
assert int(transformers.__version__.split(".")[0]) >= 5, "the Cohere models need transformers >= 5.15"

In [ ]:
from auditkit import Sample
from auditkit.registry import METRICS
kw = {"dtype": DTYPE, **({"token": os.environ["HF_TOKEN"]} if os.environ.get("HF_TOKEN") else {})}
model = HFGenModel(MODEL, device=DEVICE, name=f"hf:{MODEL}", **kw)

def run(samples, scorers, title, max_tokens=MAX_TOKENS):
    r = ak.evaluate(samples, model=model, scorers=scorers, config=RunConfig(max_tokens=max_tokens, temperature=0.0))
    print(f"=== {title} ===")
    print("headline:", {k: round(v, 3) for k, v in sorted(r.headline.items())})
    if r.errors:
        print("errors:", r.errors[:3])
    for p in r.predictions:
        print(f"  target={p.expected!r:28.28s} output={(p.raw_output or '').strip()[:70]!r}")
    return r
RESULTS = {}

## 1 · Multilingual short answers

In [ ]:
SHORT = [("What is the capital of Canada? Answer with one word.", "Ottawa"),
         ("¿Cuál es la capital de Argentina? Responde con una palabra.", "Buenos Aires"),
         ("Quelle est la couleur du ciel par temps clair ? Un mot.", "bleu"),
         ("Wie viele Tage hat eine Woche? Nur die Zahl.", "7"),
         ("भारत की राजधानी क्या है? केवल नाम लिखें।", "नई दिल्ली"),
         ("日本の首都はどこですか？一語で答えてください。", "東京")]
RESULTS["short answers"] = run([Sample(input=q, target=a) for q, a in SHORT],
                               ["exact_match", "quasi_exact_match", "f1_score", "levenshtein"], "short answers")

## 2 · Translation

In [ ]:
TRANSLATE = [("Translate to French. Reply with the translation only: The weather is nice today.", "Il fait beau aujourd'hui."),
             ("Translate to Spanish. Reply with the translation only: Where is the train station?", "¿Dónde está la estación de tren?"),
             ("Translate to German. Reply with the translation only: I would like a cup of coffee.", "Ich hätte gern eine Tasse Kaffee."),
             ("Translate to Italian. Reply with the translation only: Thank you very much for your help.", "Grazie mille per il tuo aiuto."),
             ("Translate to Portuguese. Reply with the translation only: The book is on the table.", "O livro está na mesa.")]
scorers = ["bleu", "chrf", "rouge_l", "token_overlap"] + (["bert_score"] if USE_BERT_SCORE else [])
RESULTS["translation"] = run([Sample(input=q, target=a) for q, a in TRANSLATE], scorers, "translation")

## 3 · Format following

These metrics check the *shape* of the reply, with no reference answer needed. Metrics that take a parameter are built from the registry.

In [ ]:
# one check per sample: the metric that fits what that prompt asked for
FORMAT = [
    (Sample(id="json", input='Return a JSON object with keys "name" and "age" for a 30-year-old named Ana. Output only the JSON.'),
     METRICS.get("is_json")(require_keys=["name", "age"])),
    (Sample(id="date", input="Write the date 5 March 2026 as YYYY-MM-DD. Output only the date."),
     METRICS.get("regex")(pattern=r"\b2026-03-05\b")),
    (Sample(id="three_words", input="Describe the sea in exactly three words. Output only the three words."),
     METRICS.get("word_count")(min_words=3, max_words=3)),
    (Sample(id="starts_yes", input="Is water wet? Start your answer with 'Yes' or 'No'."),
     METRICS.get("starts_with")(prefix="Yes", ignore_case=True)),
]
fmt_scores = {}
for sample, metric in FORMAT:
    r = ak.evaluate([sample], model=model, scorers=[metric], config=RunConfig(max_tokens=MAX_TOKENS, temperature=0.0))
    fmt_scores.update(r.headline)
    print(f"  {sample.id:12s} {r.headline}  <- {(r.predictions[0].raw_output or '').strip()[:70]!r}")
RESULTS["format"] = r
print("format checks:", fmt_scores)

## 4 · GSM8K slice with answer extraction

The model reasons in free text, so the metric extracts the **last number** in the reply and compares it with the gold answer (the part after `####`). This shows how to write a task-specific metric in a few lines.

In [ ]:
import re
from datasets import load_dataset
from auditkit.metric import Metric
from auditkit.score import Score
from auditkit.types import Direction, ScoreKind

class LastNumberMatch(Metric):
    name = "last_number_match"
    kind = ScoreKind.BENCHMARK
    direction = Direction.MAXIMIZE
    def score(self, sample, output, context=None):
        nums = re.findall(r"-?\d[\d,]*\.?\d*", output or "")
        pred = nums[-1].replace(",", "").rstrip(".") if nums else None
        gold = str(sample.target).replace(",", "")
        ok = pred is not None and float(pred) == float(gold)
        return Score(name=self.name, value=1.0 if ok else 0.0, kind=self.kind, metadata={"extracted": pred})

gsm = load_dataset("openai/gsm8k", "main", split=f"test[:{GSM8K_N}]")
gsm_samples = [Sample(id=f"gsm{i}", input=row["question"] + "\nThink step by step, then give the final answer as a number on the last line.",
                      target=row["answer"].split("####")[-1].strip()) for i, row in enumerate(gsm)]
RESULTS["gsm8k"] = run(gsm_samples, [LastNumberMatch()], f"GSM8K ({GSM8K_N})", max_tokens=384)

## 5 · Performance

Every run records latency and throughput (`RunConfig.track_performance`, on by default). `RunResult.summary()` prints them, and `model_size` reports the parameter count.

In [ ]:
for name, r in RESULTS.items():
    print(f"--- {name}")
    print(r.summary())
print("\nmodel size:", RESULTS["short answers"].model_size)